In [43]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
print(pd.__version__)
print(np.__version__)

2.3.3
2.3.5


In [44]:
cft_pdt_data = pd.read_csv("counterfeit_products_balanced_preprocessed_v2.csv")
cft_pdt_data.shape

(5904, 28)

In [45]:
indep_X = cft_pdt_data.drop('is_counterfeit', axis=1)
dep_Y = cft_pdt_data['is_counterfeit']

In [46]:
dep_Y.value_counts()

is_counterfeit
0    2952
1    2952
Name: count, dtype: int64

In [47]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size=0.25, random_state=0)
X_train.shape, X_test.shape

((4428, 27), (1476, 27))

In [48]:
from sklearn.feature_selection import SelectKBest, chi2
selector = SelectKBest(score_func=chi2, k=10)
X_train_set = selector.fit_transform(X_train, y_train)
X_test_set = selector.transform(X_test)

selected_cols = indep_X.columns[selector.get_support()]
print(list(selected_cols))

['price', 'seller_reviews', 'product_images', 'description_length', 'shipping_time_days', 'domain_age_days', 'views', 'purchases', 'warranty_months', 'brand_suspicious']


In [49]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train_set)
X_test_scaled = sc.transform(X_test_set)

In [50]:
from sklearn.ensemble import RandomForestClassifier

In [51]:
from sklearn.model_selection import GridSearchCV
param_grid = {'n_estimators': [50, 100, 200],'max_depth': [None, 5, 10, 20],'min_samples_split': [2, 5, 10],'criterion': ['gini', 'entropy']}

rf = RandomForestClassifier(random_state=0)
grid = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Params: {'criterion': 'gini', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
Best CV Score: 0.7380296131920266


In [52]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
best_rf = grid.best_estimator_
y_pred = best_rf.predict(X_test_scaled)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Test Accuracy: 0.7533875338753387

Confusion Matrix:
 [[609 164]
 [200 503]]

Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.79      0.77       773
           1       0.75      0.72      0.73       703

    accuracy                           0.75      1476
   macro avg       0.75      0.75      0.75      1476
weighted avg       0.75      0.75      0.75      1476



In [53]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(X_test_scaled)[:,1])

0.8270358599901733

In [54]:
table=pd.DataFrame.from_dict(grid.cv_results_)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,param_min_samples_split,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.878160,0.038479,0.018302,0.002030,gini,None,2,50,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.723476,0.747178,0.732506,0.703955,0.746893,0.730802,0.016152,19
1,1.585744,0.088427,0.030731,0.001321,gini,None,2,100,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.748307,0.729120,0.726862,0.720904,0.741243,0.733287,0.010009,8
2,2.971815,0.065118,0.054019,0.000970,gini,None,2,200,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.746050,0.740406,0.729120,0.725424,0.731073,0.734415,0.007633,4
3,0.705509,0.013138,0.014403,0.000172,gini,None,5,50,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.733634,0.733634,0.721219,0.726554,0.738983,0.730805,0.006211,18
4,1.404533,0.015766,0.028963,0.003065,gini,None,5,100,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.735892,0.740406,0.717833,0.719774,0.738983,0.730578,0.009743,20
5,2.823634,0.041552,0.054497,0.004492,gini,None,5,200,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.734763,0.747178,0.713318,0.717514,0.734463,0.729447,0.012413,26
6,0.666024,0.009932,0.013791,0.000280,gini,None,10,50,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.734763,0.734763,0.708804,0.709605,0.732203,0.724027,0.012142,41
7,1.368820,0.016558,0.025374,0.000756,gini,None,10,100,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.731377,0.735892,0.714447,0.718644,0.727684,0.725609,0.007955,32
8,2.730653,0.047596,0.050599,0.001225,gini,None,10,200,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.734763,0.733634,0.716704,0.719774,0.738983,0.728772,0.008836,29
9,0.379931,0.009468,0.009730,0.000160,gini,5,2,50,"{'criterion': 'gini', 'max_depth': 5, 'min_sam...",0.699774,0.712190,0.706546,0.706215,0.723164,0.709578,0.007849,60


In [57]:
import pickle

pickle.dump(best_rf, open("counterfeit_model.sav", "wb"))
pickle.dump(sc, open("scaler.sav", "wb"))
pickle.dump(list(selected_cols), open("selected_features.sav", "wb"))

print("All 3 files saved separately!")

All 3 files saved separately!
